### Imports and Load

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split

import shutil

artifacts_path = Path.cwd().parent.parent / "artifacts"
artifacts_path.mkdir(parents=True, exist_ok=True)


# Paths
PROCESSED_PATH = Path.cwd().parent.parent / "data" / "processed"
OUTPUT_PATH    = Path.cwd().parent.parent / "data" / "model-live" / "scaled"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
artifacts_path = Path.cwd().parent.parent / "artifacts"
artifacts_path.mkdir(parents=True, exist_ok=True)

# Column config
TARGET_COL    = "label"
METADATA_COLS = ["scenario_id", "timestep"]
NON_ML_COLS   = ["fault_type", "effect_factor"]

# Split config
RANDOM_SEED = 42

# Scalers to compare
SCALERS = {
    "robust":   RobustScaler(),
    "standard": StandardScaler(),
}

In [2]:
def load_data(path: Path, filename: str) -> pd.DataFrame:
    df = pd.read_csv(path / filename)
    print(f"Loaded shape : {df.shape}")
    print(f"Label distribution:\n{df[TARGET_COL].value_counts().to_string()}\n")
    return df

df = load_data(PROCESSED_PATH, "live_feature_dataset.csv")
df.head()

Loaded shape : (151416, 54)
Label distribution:
label
2    50472
1    50472
0    50472



,scenario_id,timestep,label,node_a_pressure,node_b_pressure,node_c_pressure,velocity_a,velocity_b,velocity_c,pressure_drop_ab,...,rolling_std_velocity_c,rolling_std_pressure_drop_ab,rolling_std_pressure_drop_bc,rolling_std_midpoint_pressure_deviation,rolling_std_midpoint_velocity_deviation,roc_node_a_pressure,roc_node_b_pressure,roc_node_c_pressure,roc_velocity_b,roc_velocity_c
0,blockage_25_run0,0,2,0.000000,0.000000,0.000000,2.951007,2.951007,2.951007,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,blockage_25_run0,1,2,117289.689466,57665.388553,245.993913,4.001445,3.001142,3.001098,59624.300913,...,0.035420,42160.747499,40601.643322,779.552089,0.353645,117289.689466,57665.388553,245.993913,0.050135,0.050091
2,blockage_25_run0,2,2,115089.912449,56606.885212,238.286542,4.001398,3.001385,3.001193,58483.027237,...,0.028948,34099.423846,32851.965870,623.852482,0.288687,-2199.777017,-1058.503341,-7.707371,0.000243,0.000095
3,blockage_25_run0,3,2,113035.851969,55594.219694,231.778470,4.001781,3.000838,3.001208,57441.632275,...,0.025080,29271.735377,28204.245067,533.867327,0.250116,-2054.060480,-1012.665518,-6.508072,-0.000547,0.000015
4,blockage_25_run0,4,2,111099.763026,54644.155216,224.895595,4.001291,3.001167,3.000970,56455.607810,...,0.022410,25965.720524,25020.890691,472.542925,0.223681,-1936.088943,-950.064478,-6.882875,0.000329,-0.000238


In [3]:
df.columns

Index(['scenario_id', 'timestep', 'label', 'node_a_pressure',
       'node_b_pressure', 'node_c_pressure', 'velocity_a', 'velocity_b',
       'velocity_c', 'pressure_drop_ab', 'pressure_drop_bc',
       'pressure_drop_ac', 'grad_ab', 'grad_bc', 'grad_ac', 'dev_p_a',
       'dev_p_b', 'dev_p_c', 'dev_v_b', 'dev_v_c', 'vel_drop_ab',
       'vel_drop_bc', 'vel_drop_ac', 'midpoint_pressure_deviation',
       'midpoint_velocity_deviation', 'grad_ratio_ab_bc', 'p_ratio_ba',
       'p_ratio_ca', 'p_ratio_cb', 'vp_coupling_b', 'vp_coupling_c',
       'rolling_mean_node_a_pressure', 'rolling_mean_node_b_pressure',
       'rolling_mean_node_c_pressure', 'rolling_mean_velocity_b',
       'rolling_mean_velocity_c', 'rolling_mean_pressure_drop_ab',
       'rolling_mean_pressure_drop_bc',
       'rolling_mean_midpoint_pressure_deviation',
       'rolling_mean_midpoint_velocity_deviation',
       'rolling_std_node_a_pressure', 'rolling_std_node_b_pressure',
       'rolling_std_node_c_pressure', 'roll

### Define Columns

In [4]:
def get_feature_columns(df: pd.DataFrame) -> list[str]:
    exclude = set(METADATA_COLS + NON_ML_COLS + [TARGET_COL])
    feature_cols = [c for c in df.columns if c not in exclude]
    print(f"Total ML features : {len(feature_cols)}")
    print(f"Excluded          : {sorted(exclude)}")
    return feature_cols

feature_cols = get_feature_columns(df)

Total ML features : 51
Excluded          : ['effect_factor', 'fault_type', 'label', 'scenario_id', 'timestep']


### Scenario Level Split


In [5]:
def split_scenario_ids(scenarios, seed=RANDOM_SEED):
    """75 / 12.5 / 12.5 split on scenario IDs — NOT on rows."""
    train, temp = train_test_split(scenarios, test_size=0.25, random_state=seed)
    val, test   = train_test_split(temp,      test_size=0.50, random_state=seed)
    return train, val, test

def scenario_level_split(df: pd.DataFrame):
    train_ids, val_ids, test_ids = [], [], []

    for label in sorted(df[TARGET_COL].unique()):
        scenarios = df[df[TARGET_COL] == label]["scenario_id"].unique()
        tr, va, te = split_scenario_ids(scenarios)
        train_ids.extend(tr); val_ids.extend(va); test_ids.extend(te)

    df_train = df[df["scenario_id"].isin(train_ids)].copy()
    df_val   = df[df["scenario_id"].isin(val_ids)].copy()
    df_test  = df[df["scenario_id"].isin(test_ids)].copy()

    for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
        print(f"{name:5s} → shape: {split.shape} | labels: {split[TARGET_COL].value_counts().to_dict()}")

    return df_train, df_val, df_test

df_train, df_val, df_test = scenario_level_split(df)

Train → shape: (113562, 54) | labels: {2: 37854, 1: 37854, 0: 37854}
Val   → shape: (18927, 54) | labels: {2: 6309, 1: 6309, 0: 6309}
Test  → shape: (18927, 54) | labels: {2: 6309, 1: 6309, 0: 6309}


- We are splittitng by scenario and not random rows because if we are to split randomly timesteps 1-699 of a scenario end up in train and 700 ends up in testing.
- The model basically would have een the scansion during training leading to data leakage.
- Scenario level split guarantees all 700 timesteps of a scenario stay in the same split.

In [6]:
# Save metadata
df_train[["scenario_id", "timestep"]].to_csv(OUTPUT_PATH / "train_meta.csv", index=False)
df_val[["scenario_id", "timestep"]].to_csv(OUTPUT_PATH / "val_meta.csv", index=False)
df_test[["scenario_id", "timestep"]].to_csv(OUTPUT_PATH / "test_meta.csv", index=False)

### Extracting features and targets

In [7]:
def extract_Xy(df_train, df_val, df_test, feature_cols):
    X_train = df_train[feature_cols].copy()
    X_val   = df_val[feature_cols].copy()
    X_test  = df_test[feature_cols].copy()

    y_train = df_train[TARGET_COL].copy()
    y_val   = df_val[TARGET_COL].copy()
    y_test  = df_test[TARGET_COL].copy()

    print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = extract_Xy(df_train, df_val, df_test, feature_cols)

X_train: (113562, 51) | X_val: (18927, 51) | X_test: (18927, 51)


### Clean Features

In [8]:
def clean_features(X_train, X_val, X_test):
    print(f"Inf values in X_train : {np.isinf(X_train.select_dtypes(include=np.number)).sum().sum()}")
    print(f"NaN values in X_train : {X_train.isnull().sum().sum()}")

    cleaned = []
    for X in [X_train, X_val, X_test]:
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        X = X.clip(lower=X.quantile(0.001), upper=X.quantile(0.999), axis=1)
        cleaned.append(X)

    print("Clean-up done.\n")
    return cleaned[0], cleaned[1], cleaned[2]

X_train, X_val, X_test = clean_features(X_train, X_val, X_test)

Inf values in X_train : 0
NaN values in X_train : 0
Clean-up done.



### Scale and Save

In [9]:
def scale_and_save(scaler_name, scaler, X_train, X_val, X_test, feature_cols, save_y=False):
    # Fit on train only, transform all splits
    X_tr = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
    X_va = pd.DataFrame(scaler.transform(X_val),       columns=feature_cols, index=X_val.index)
    X_te = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols, index=X_test.index)

    # Save scaled CSVs
    X_tr.to_csv(OUTPUT_PATH / f"X_train_{scaler_name}_live_data.csv", index=False)
    X_va.to_csv(OUTPUT_PATH / f"X_val_{scaler_name}_live_data.csv",   index=False)
    X_te.to_csv(OUTPUT_PATH / f"X_test_{scaler_name}_live_data.csv",  index=False)

    # Save fitted scaler — using your exact naming convention
    scaler_filename = f"robust_scaler_live.joblib" if scaler_name == "robust" else f"standard_scaler_live.joblib"
    joblib.dump(scaler, artifacts_path / scaler_filename)

    # Verify it loads back cleanly
    loaded = joblib.load(artifacts_path/ scaler_filename)
    n_features = len(loaded.scale_) if hasattr(loaded, "scale_") else len(loaded.mean_)
    print(f"\n── {scaler_name.upper()} SCALER")
    print(f"Saved & verified — {n_features} features")
    print(f"Scaler path : {artifacts_path/ scaler_filename}")
    print(f"X_train     : {X_tr.shape} | X_val: {X_va.shape} | X_test: {X_te.shape}")

    # Sanity check on a sample column
    s = X_tr["node_b_pressure"]
    print(f"node_b_pressure → mean={s.mean():.4f}  median={s.median():.4f}  std={s.std():.4f}  min={s.min():.4f}  max={s.max():.4f}")

    return X_tr, X_va, X_te

# Save y splits once (scaler-independent)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
y_train.to_csv(OUTPUT_PATH / "y_train_live.csv", index=False)
y_val.to_csv(  OUTPUT_PATH / "y_val_live.csv",   index=False)
y_test.to_csv( OUTPUT_PATH / "y_test_live.csv",  index=False)

# Run both scalers
scaled_results = {}
for scaler_name, scaler in SCALERS.items():
    X_tr_sc, X_va_sc, X_te_sc = scale_and_save(
        scaler_name, scaler, X_train, X_val, X_test, feature_cols
    )
    scaled_results[scaler_name] = (X_tr_sc, X_va_sc, X_te_sc)

print("\nPREPROCESSING COMPLETE — both scalers saved and ready.")


── ROBUST SCALER
Saved & verified — 51 features
Scaler path : /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/artifacts/robust_scaler_live.joblib
X_train     : (113562, 51) | X_val: (18927, 51) | X_test: (18927, 51)
node_b_pressure → mean=3.7114  median=0.0000  std=12.6728  min=-30.5199  max=66.4613

── STANDARD SCALER
Saved & verified — 51 features
Scaler path : /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/artifacts/standard_scaler_live.joblib
X_train     : (113562, 51) | X_val: (18927, 51) | X_test: (18927, 51)
node_b_pressure → mean=-0.0000  median=-0.2929  std=1.0000  min=-2.7012  max=4.9516

PREPROCESSING COMPLETE — both scalers saved and ready.
